In [2]:
!pip install -q -U transformers datasets accelerate scikit-learn

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import numpy as np
import torch
from sklearn.metrics import accuracy_score


# 1. Load IMDB dataset
print("Loading IMDB dataset...")

dataset = load_dataset("stanfordnlp/imdb")

small_train = dataset["train"].shuffle(seed=42).select(range(2000))
small_test = dataset["test"].shuffle(seed=42).select(range(500))

print("Training samples:", len(small_train))
print("Testing samples:", len(small_test))


# 2. Load tokenizer
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)


# 3. Tokenization
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


print("Tokenizing dataset...")

train_ds = small_train.map(tokenize, batched=True)
test_ds = small_test.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])


# 4. Load pre-trained DistilBERT
print("\nLoading DistilBERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


# 5. Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none"
)


# 6. Accuracy metric
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }


# 7. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)


# 8. Train
print("\nStarting training...\n")

trainer.train()


# 9. Evaluate
print("\nEvaluating model...")

metrics = trainer.evaluate()

print("\nEvaluation Metrics:")
print(metrics)


# 10. Save model and tokenizer
save_path = "./fine_tuned_distilbert_imdb"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("\nModel saved successfully!")
print("Location:", save_path)


# 11. Test with a new review
print("\nTesting the fine-tuned model...")

test_review = "This movie was amazing and I really enjoyed it!"

inputs = tokenizer(
    test_review,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=128
)

# Move inputs to the same device as the model
device = next(model.parameters()).device
inputs = {key: value.to(device) for key, value in inputs.items()}

# Make prediction
with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

if prediction == 1:
    sentiment = "Positive"
else:
    sentiment = "Negative"

print("\nReview:", test_review)
print("Predicted Sentiment:", sentiment)

Loading IMDB dataset...
Training samples: 2000
Testing samples: 500

Loading tokenizer...
Tokenizing dataset...

Loading DistilBERT model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting training...



Epoch,Training Loss,Validation Loss,Accuracy
1,0.372580,0.485319,0.798000
2,0.270642,0.620927,0.814000


...


Evaluating model...


Training Loss,Validation Loss,Epoch,Accuracy
0.270642,0.485319,2,0.798000



Evaluation Metrics:
{'eval_loss': 0.48531922698020935, 'eval_accuracy': 0.798}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved successfully!
Location: ./fine_tuned_distilbert_imdb

Testing the fine-tuned model...

Review: This movie was amazing and I really enjoyed it!
Predicted Sentiment: Positive
